In [1]:
#Loading synthetic data with real world data

import pandas as pd

train_df = pd.read_csv(r"../data\synthetic_maternal_cleaned.csv")

test_df = pd.read_csv(r"../data\Maternal Health Risk Data Set.csv")

In [2]:
# Standardize RiskLevel labels in both datasets
train_df["RiskLevel"] = train_df["RiskLevel"].str.replace("_", " ").str.lower().str.strip()
test_df["RiskLevel"] = test_df["RiskLevel"].str.replace("_", " ").str.lower().str.strip()


In [3]:
#Feature Engineering on training data

# Feature Engineering on training data
train_df["MAP"] = (train_df["SystolicBP"] + 2 * train_df["DiastolicBP"]) / 3
train_df["PulsePressure"] = train_df["SystolicBP"] - train_df["DiastolicBP"]
train_df["ShockIndex"] = train_df["HeartRate"] / train_df["SystolicBP"]
train_df["BPRatio"] = train_df["SystolicBP"] / train_df["DiastolicBP"]

train_df["TempDeviation"] = train_df["BodyTemp"] - 98.2
train_df["BSDeviation"] = train_df["BS"] - 7

train_df["CombinedRiskScore"] = (
    (train_df["MAP"] > 105).astype(int) +
    (train_df["BS"] > 10).astype(int) +
    (train_df["HeartRate"] > 90).astype(int) +
    (train_df["TempDeviation"] > 1).astype(int)
)


In [4]:
#Feature engineering on testing data

# Feature Engineering on test data
test_df["MAP"] = (test_df["SystolicBP"] + 2 * test_df["DiastolicBP"]) / 3
test_df["PulsePressure"] = test_df["SystolicBP"] - test_df["DiastolicBP"]
test_df["ShockIndex"] = test_df["HeartRate"] / test_df["SystolicBP"]
test_df["BPRatio"] = test_df["SystolicBP"] / test_df["DiastolicBP"]

test_df["TempDeviation"] = test_df["BodyTemp"] - 98.2
test_df["BSDeviation"] = test_df["BS"] - 7

test_df["CombinedRiskScore"] = (
    (test_df["MAP"] > 105).astype(int) +
    (test_df["BS"] > 10).astype(int) +
    (test_df["HeartRate"] > 90).astype(int) +
    (test_df["TempDeviation"] > 1).astype(int)
)


In [5]:
#Preparing X and y
X_train = train_df.drop("RiskLevel" , axis = 1)
y_train = train_df["RiskLevel"]

In [6]:
#Real(Evaluation)
X_test = test_df.drop("RiskLevel", axis=1)
y_test = test_df["RiskLevel"]

In [7]:
#Encode target and Scale features

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
#Training single models: 
#1. Logistic Regression
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter = 2000, multi_class="multinomial")
lr.fit(X_train_scaled, y_train_enc)

c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'multinomial'


In [9]:
#2. Decision Tree

from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train_enc)

,criterion,'gini'
,splitter,'best'
,max_depth,6
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [10]:
#3 Random Forest

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)
rf.fit(X_train, y_train_enc)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
#4. Gradient Boosting

from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train_enc)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [12]:
#Voting Ensemble to average the predicted prob of each model

from sklearn.ensemble import VotingClassifier

voting = VotingClassifier(
    estimators=[
        ('lr',lr),
        ('rf', rf),
        ('gb',gb)
    ],
    voting='soft'
)

voting.fit(X_train_scaled, y_train_enc)

c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,estimators,"[('lr', ...), ('rf', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True


In [13]:
#Stacking Ensemble

from sklearn.ensemble import StackingClassifier

stack = StackingClassifier(
    estimators=[
        ('lr', lr),
        ('rf',rf),
        ('gb',gb)
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    passthrough=False
)
stack.fit(X_train_scaled, y_train_enc)

c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_log

,estimators,"[('lr', ...), ('rf', ...), ...]"
,final_estimator,LogisticRegre...max_iter=2000)
,cv,None
,stack_method,'auto'
,n_jobs,None
,passthrough,False
,verbose,0
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


In [14]:
#MODEL EVAL

from sklearn.metrics import recall_score, classification_report

def evaluate_recall(model, X_test, y_test, name):
    print(f"\n===== {name} — Recall =====")
    y_pred = model.predict(X_test)

    # Macro recall = average recall across all classes
    macro_recall = recall_score(y_test, y_pred, average='macro')

    # Weighted recall = weighted by class frequency
    weighted_recall = recall_score(y_test, y_pred, average='weighted')

    print("Macro Recall:", macro_recall)
    print("Weighted Recall:", weighted_recall)
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    return macro_recall, weighted_recall


In [15]:
rec_lr, wrec_lr = evaluate_recall(lr, X_test_scaled, y_test_enc, "Logistic Regression")
rec_dt, wrec_dt = evaluate_recall(dt, X_test, y_test_enc, "Decision Tree")
rec_rf, wrec_rf = evaluate_recall(rf, X_test, y_test_enc, "Random Forest")
rec_gb, wrec_gb = evaluate_recall(gb, X_test, y_test_enc, "Gradient Boosting")
rec_vote, wrec_vote = evaluate_recall(voting, X_test_scaled, y_test_enc, "Voting Ensemble")
rec_stack, wrec_stack = evaluate_recall(stack, X_test_scaled, y_test_enc, "Stacking Ensemble")



===== Logistic Regression — Recall =====
Macro Recall: 0.3333333333333333
Weighted Recall: 0.40039447731755423
              precision    recall  f1-score   support

   high risk       0.00      0.00      0.00       272
    low risk       0.42      1.00      0.59       406
    mid risk       0.00      0.00      0.00       336

    accuracy                           0.40      1014
   macro avg       0.14      0.33      0.20      1014
weighted avg       0.17      0.40      0.24      1014


===== Decision Tree — Recall =====
Macro Recall: 0.45849029266879165
Weighted Recall: 0.4930966469428008
              precision    recall  f1-score   support

   high risk       0.94      0.22      0.35       272
    low risk       0.59      0.74      0.66       406
    mid risk       0.32      0.42      0.36       336

    accuracy                           0.49      1014
   macro avg       0.62      0.46      0.46      1014
weighted avg       0.59      0.49      0.48      1014


===== Random Forest

c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [16]:
# Voting Ensemble Recall
rec_vote, wrec_vote = evaluate_recall(
    voting, 
    X_test_scaled, 
    y_test_enc, 
    "Voting Ensemble"
)

# Stacking Ensemble Recall
rec_stack, wrec_stack = evaluate_recall(
    stack, 
    X_test_scaled, 
    y_test_enc, 
    "Stacking Ensemble"
)



===== Voting Ensemble — Recall =====
Macro Recall: 0.4488855887182459
Weighted Recall: 0.5
              precision    recall  f1-score   support

   high risk       1.00      0.13      0.23       272
    low risk       0.57      0.90      0.70       406
    mid risk       0.31      0.32      0.32       336

    accuracy                           0.50      1014
   macro avg       0.63      0.45      0.41      1014
weighted avg       0.60      0.50      0.45      1014


===== Stacking Ensemble — Recall =====
Macro Recall: 0.4168115522070897
Weighted Recall: 0.4723865877712032
              precision    recall  f1-score   support

   high risk       1.00      0.04      0.07       272
    low risk       0.57      0.87      0.69       406
    mid risk       0.30      0.34      0.32       336

    accuracy                           0.47      1014
   macro avg       0.62      0.42      0.36      1014
weighted avg       0.60      0.47      0.40      1014



In [17]:
recall_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "Voting Ensemble",
        "Stacking Ensemble"
    ],
    "Macro Recall": [
        rec_lr,
        rec_dt,
        rec_rf,
        rec_gb,
        rec_vote,
        rec_stack
    ],
    "Weighted Recall": [
        wrec_lr,
        wrec_dt,
        wrec_rf,
        wrec_gb,
        wrec_vote,
        wrec_stack
    ]
})

recall_results.sort_values(by="Macro Recall", ascending=False)


,Model,Macro Recall,Weighted Recall
1,Decision Tree,0.458490,0.493097
3,Gradient Boosting,0.458490,0.493097
4,Voting Ensemble,0.448886,0.500000
2,Random Forest,0.416812,0.472387
5,Stacking Ensemble,0.416812,0.472387
0,Logistic Regression,0.333333,0.400394


In [18]:
#Trying XGB and Light GBM to handle imbalanced data
#Training XGBoost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss'
)

xgb.fit(X_train_scaled, y_train_enc)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [19]:
from lightgbm import LGBMClassifier

lgb = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight='balanced',
    objective='multiclass',
    num_class=3
)

lgb.fit(X_train_scaled, y_train_enc)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000518 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1717
[LightGBM] [Info] Number of data points in the train set: 1589, number of used features: 13
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further spli

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,'multiclass'
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [20]:
rec_xgb, wrec_xgb = evaluate_recall(xgb, X_test_scaled, y_test_enc, "XGBoost")
rec_lgb, wrec_lgb = evaluate_recall(lgb, X_test_scaled, y_test_enc, "LightGBM")



===== XGBoost — Recall =====
Macro Recall: 0.4234058565955117
Weighted Recall: 0.47731755424063116
              precision    recall  f1-score   support

   high risk       1.00      0.06      0.12       272
    low risk       0.57      0.87      0.69       406
    mid risk       0.30      0.33      0.31       336

    accuracy                           0.48      1014
   macro avg       0.62      0.42      0.37      1014
weighted avg       0.60      0.48      0.41      1014


===== LightGBM — Recall =====
Macro Recall: 0.44913712611481377
Weighted Recall: 0.48619329388560156
              precision    recall  f1-score   support

   high risk       0.94      0.25      0.39       272
    low risk       0.53      0.80      0.63       406
    mid risk       0.31      0.30      0.31       336

    accuracy                           0.49      1014
   macro avg       0.59      0.45      0.44      1014
weighted avg       0.57      0.49      0.46      1014



c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [21]:
#Adding to comparison table
recall_results.loc[len(recall_results)] = ["XGBoost", rec_xgb, wrec_xgb]
recall_results.loc[len(recall_results)] = ["LightGBM", rec_lgb, wrec_lgb]

recall_results.sort_values(by="Macro Recall", ascending=False)


,Model,Macro Recall,Weighted Recall
1,Decision Tree,0.458490,0.493097
3,Gradient Boosting,0.458490,0.493097
7,LightGBM,0.449137,0.486193
4,Voting Ensemble,0.448886,0.500000
6,XGBoost,0.423406,0.477318
2,Random Forest,0.416812,0.472387
5,Stacking Ensemble,0.416812,0.472387
0,Logistic Regression,0.333333,0.400394


In [28]:
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV



num_classes = len(set(y_train_enc))

pipeline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42, k_neighbors=5)),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    ))
])


In [29]:
param_grid = {
    "xgb__max_depth": [3, 4, 5],
    "xgb__learning_rate": [0.01, 0.05, 0.1],
    "xgb__n_estimators": [200, 400, 600],
    "xgb__subsample": [0.8, 1.0],
    "xgb__colsample_bytree": [0.8, 1.0],
    "xgb__gamma": [0, 1, 5]
}


In [30]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="recall_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)


In [31]:
grid.fit(X_train, y_train_enc)


Fitting 5 folds for each of 324 candidates, totalling 1620 fits


,estimator,"Pipeline(step...ass=3, ...))])"
,param_grid,"{'xgb__colsample_bytree': [0.8, 1.0], 'xgb__gamma': [0, 1, ...], 'xgb__learning_rate': [0.01, 0.05, ...], 'xgb__max_depth': [3, 4, ...], ...}"
,scoring,'recall_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,copy,True


In [32]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test_enc, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.11      0.19       272
           1       0.58      0.91      0.71       406
           2       0.31      0.32      0.31       336

    accuracy                           0.50      1014
   macro avg       0.63      0.44      0.40      1014
weighted avg       0.60      0.50      0.44      1014



In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
}


In [34]:
models["XGBoost (SMOTE)"] = best_model   # best_model from GridSearchCV


In [35]:
estimators = [
    ("lr", LogisticRegression(max_iter=2000)),
    ("dt", DecisionTreeClassifier()),
    ("rf", RandomForestClassifier()),
    ("gb", GradientBoostingClassifier())
]

models["Voting Ensemble"] = VotingClassifier(
    estimators=estimators,
    voting="soft"
)

models["Stacking Ensemble"] = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=2000)
)


In [36]:
from sklearn.metrics import recall_score

results = {}

for name, model in models.items():
    model.fit(X_train, y_train_enc)
    preds = model.predict(X_test)
    macro_recall = recall_score(y_test_enc, preds, average="macro")
    results[name] = macro_recall


c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\shiro\anaconda3\envs\ML_projects\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/mod

In [37]:
import pandas as pd

comparison_df = pd.DataFrame.from_dict(results, orient="index", columns=["Macro Recall"])
comparison_df = comparison_df.sort_values(by="Macro Recall", ascending=False)

comparison_df


,Macro Recall
Decision Tree,0.458490
Gradient Boosting,0.458490
Voting Ensemble,0.456039
Random Forest,0.443772
XGBoost (SMOTE),0.443654
Stacking Ensemble,0.405782
Logistic Regression,0.333333


---
# Final Model Performance & Evaluation Summary
The following sections provide the definitive evaluation of our trained models, as generated by the master pipeline.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Set plot style for premium aesthetics
sns.set_theme(style="whitegrid", palette="magma")

# Load and Preprocess (matching the Master Pipeline logic)
data_path = os.path.join("..", "data", "Maternal Health Risk Data Set.csv")
df = pd.read_csv(data_path)

# Feature Engineering
df["MAP"] = (df["SystolicBP"].astype(float) + 2 * df["DiastolicBP"].astype(float)) / 3
df["PulsePressure"] = df["SystolicBP"] - df["DiastolicBP"]
df["ShockIndex"] = df["HeartRate"] / df["SystolicBP"]
df["BPRatio"] = df["SystolicBP"] / df["DiastolicBP"]
temp_dev = df["BodyTemp"] - 98.2
df["CombinedRiskScore"] = (
    (df["MAP"].astype(float) > 105).astype(int) +
    (df["BS"].astype(float) > 10).astype(int) +
    (df["HeartRate"].astype(float) > 90).astype(int) +
    (temp_dev > 1).astype(int)
)

# Prepare for modeling
le = LabelEncoder()
y = le.fit_transform(df['RiskLevel'])
X = df.drop('RiskLevel', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Setup Complete: Data loaded, engineered, and balanced with SMOTE.")

## 6. Model Evaluation: Confusion Matrix

To conclude our analysis, we evaluate how a baseline model (Random Forest) performs on the dataset after balancing classes with SMOTE. The confusion matrix below shows where the model excels and where it confuses risk categories.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import SMOTE

# 1. Prepare data (using the engineered df_encoded from previous steps)
# We drop the target variable 'RiskLevel' to create X
X = df_encoded.drop('RiskLevel', axis=1)
y = df_encoded['RiskLevel']

# 2. Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Apply SMOTE to handle imbalance
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 4. Train Random Forest (Champion Model)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_res, y_train_res)

# 5. Generate and Plot Confusion Matrix
y_pred = rf.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Low', 'Mid', 'High'], 
            yticklabels=['Low', 'Mid', 'High'])

plt.title('Confusion Matrix: Random Forest Performance', fontsize=15)
plt.xlabel('Predicted Risk Level', fontsize=12)
plt.ylabel('True Risk Level', fontsize=12)
plt.show()plt.savefig('../figures/confusion_matrix.png', dpi=150, bbox_inches='tight')


## 7. Feature Importance Analysis

We now examine which features contribute most to the models' predictions. This analysis includes both the original physiological readings and our engineered clinical metrics (MAP, Shock Index, etc.). Understanding feature importance helps validate the medical relevance of our engineered features.

In [ ]:
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier

# 1. Initialize models (using parameters consistent with the main pipeline)
models_to_plot = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(objective="multi:softprob", num_class=3, random_state=42, eval_metric="mlogloss"),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# 2. Train models and plot importances in a vertical stack of horizontal bar charts
fig, axes = plt.subplots(3, 1, figsize=(12, 18))
plt.subplots_adjust(hspace=0.4)

for i, (name, model) in enumerate(models_to_plot.items()):
    # Train model on the SMOTE-balanced training set from point 6
    model.fit(X_train_res, y_train_res)
    
    # Get importances and sort them
    importances = model.feature_importances_
    feature_names = X.columns
    feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
    
    # Plot using Seaborn
    sns.barplot(x='Importance', y='Feature', data=feature_importance_df, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Feature Importance: {name}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel('Importance Score', fontsize=12)
    axes[i].set_ylabel('Features', fontsize=12)
    axes[i].grid(True, linestyle='--', alpha=0.7)

plt.show()

## 8. Multiclass ROC Curves (One-vs-Rest)

The Receiver Operating Characteristic (ROC) curve illustrates the trade-off between sensitivity (True Positive Rate) and specificity (False Positive Rate) for each risk category. A higher Area Under the Curve (AUC) indicates a model that is better at distinguishing between different risk levels, which is a critical 'tool to analyse' for medical diagnostic systems.

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import VotingClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

# 1. Define the Top 3 Models (Consistent with results and previous analysis)
top_3_models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(objective="multi:softprob", num_class=3, random_state=42, eval_metric="mlogloss"),
    "Voting Ensemble": VotingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)), 
            ('rf', RandomForestClassifier(random_state=42)), 
            ('gb', GradientBoostingClassifier(random_state=42))
        ], 
        voting='soft'
    )
}

# 2. Binarize the target labels (One-vs-Rest requirement for multiclass ROC)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_bin.shape[1]
risk_labels = ['Low Risk', 'Mid Risk', 'High Risk']
colors = ['#3498db', '#e67e22', '#e74c3c']

# 3. Plotting in a 1x3 horizontal grid for direct comparison
fig, axes = plt.subplots(1, 3, figsize=(22, 6))
plt.subplots_adjust(wspace=0.3)

for i, (name, model) in enumerate(top_3_models.items()):
    model.fit(X_train_res, y_train_res)
    y_score = model.predict_proba(X_test)
    
    for j in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_bin[:, j], y_score[:, j])
        roc_auc = auc(fpr, tpr)
        axes[i].plot(fpr, tpr, color=colors[j], lw=2.5, 
                     label=f'{risk_labels[j]} (AUC = {roc_auc:.2f})')
    
    axes[i].plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--')
    axes[i].set_xlim([0.0, 1.0])
    axes[i].set_ylim([0.0, 1.05])
    axes[i].set_xlabel('False Positive Rate', fontsize=12)
    axes[i].set_ylabel('True Positive Rate', fontsize=12)
    axes[i].set_title(f'ROC Analysis: {name}', fontsize=14, fontweight='bold')
    axes[i].legend(loc="lower right", fontsize=10)
    axes[i].grid(True, linestyle='--', alpha=0.5)

plt.suptitle("Multiclass ROC Curve Comparison (One-vs-Rest)", fontsize=16, y=1.02)
plt.show()plt.savefig('../figures/roc_curves.png', dpi=150, bbox_inches='tight')


## 9. Final Model Comparison & Performance Summary

As a final step, we compare the performance of all trained models using Macro Recall. In medical diagnostics, Macro Recall is the primary metric as it ensures the model is equally capable of identifying all risk levels, particularly the 'High Risk' cases, without being biased toward the majority class.

In [ ]:
from sklearn.metrics import recall_score

# 1. Define all models to compare (consistent with the Master Pipeline)
final_models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(objective="multi:softprob", num_class=3, random_state=42, eval_metric="mlogloss"),
    "Voting Ensemble": VotingClassifier(
        estimators=[
            ('dt', DecisionTreeClassifier(random_state=42)), 
            ('rf', RandomForestClassifier(random_state=42)), 
            ('gb', GradientBoostingClassifier(random_state=42))
        ], 
        voting='soft'
    )
}

# 2. Calculate scores on the test set
results = {}
for name, model in final_models.items():
    model.fit(X_train_res, y_train_res)
    preds = model.predict(X_test)
    results[name] = recall_score(y_test, preds, average='macro')

# 3. Create a sorted DataFrame for plotting
comparison_df = pd.DataFrame.from_dict(results, orient='index', columns=['Macro Recall']).sort_values(by='Macro Recall', ascending=False)

# 4. Plotting the results section key figure
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")
ax = sns.barplot(x=comparison_df['Macro Recall'], y=comparison_df.index, palette='rocket')

# Annotate each bar with the specific score
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', padding=5, fontsize=12, fontweight='bold')

plt.title('Final Model Performance Comparison (Macro Recall)', fontsize=16, fontweight='bold')
plt.xlabel('Macro Recall Score', fontsize=13)
plt.ylabel('Machine Learning Model', fontsize=13)
plt.xlim(0, 1.0)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print("Champion Model determined to be:", comparison_df.index[0])plt.savefig('../figures/model_comparison.png', dpi=150, bbox_inches='tight')
